In [3]:
# Required packages:
# numpy, scipy, plotly
#
# If plotly is not installed, run:
# pip install plotly

import numpy as np
from scipy.optimize import minimize
import plotly.graph_objects

"""
Necker exercise:
The goal is to reconstruct a 3D structure from its 2D projection.

You are given:
- The true 3D object
- A function that projects 3D → 2D

Your task:
Implement the objective function so that it measures how well a 3D guess
matches the observed 2D projection.

See the 02464 lecture notes for details.
"""

# define viewing angles (in degrees)
azimuth = np.deg2rad(-20)
elevation = np.deg2rad(25)

# viewpoint in Cartesian coordinates
eye1 = np.array([
    np.cos(elevation) * np.cos(azimuth),
    np.cos(elevation) * np.sin(azimuth),
    np.sin(elevation)
], dtype=float)


# Define vertices of cube (8 corners)
vertices_3d_cube = np.array([
    [-1, -1, -1],
    [ 1, -1, -1],
    [ 1,  1, -1],
    [-1,  1, -1],
    [-1, -1,  1],
    [ 1, -1,  1],
    [ 1,  1,  1],
    [-1,  1,  1],
], dtype=float)

# Define edges of cube (pairs of vertex indices)
edges_cube = [
    (0,1),(1,2),(2,3),(3,0),  # bottom square
    (4,5),(5,6),(6,7),(7,4),  # top square
    (0,4),(1,5),(2,6),(3,7)   # vertical edges
]

# Define vertices of hexagonal prism (12 vertices: 6 bottom, 6 top)
vertices_3d_hexa = np.array([
    [0.0000,  0.8660,  0.8660,  0.0000, -0.8660, -0.8660,  0.0000,  0.8660,  0.8660,  0.0000, -0.8660, -0.8660],
    [1.0000,  0.5000, -0.5000, -1.0000, -0.5000,  0.5000,  1.0000,  0.5000, -0.5000, -1.0000, -0.5000,  0.5000],
    [-1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000]
], dtype=float).T

# Define edges of hexagonal prism (pairs of vertex indices)
edges_hexa = [
    (0, 1), (1, 2), (2, 3), (3, 4), (4, 5), (5, 0),
    (6, 7), (7, 8), (8, 9), (9, 10), (10, 11), (11, 6),
    (0, 6), (1, 7), (2, 8), (3, 9), (4, 10), (5, 11),
]

# select which structure to use: _hexa for hexagonal prism, _cube for cube
structure = "cube"  # "cube" or "hexa"
structures = {
    "cube": (vertices_3d_cube, edges_cube),
    "hexa": (vertices_3d_hexa, edges_hexa),
}
vertices_3d_true, edges = structures[structure]

def project_orthographic(vertices_3d, eye, center=None, up=None):
    if center is None:
        center = np.array([0.0, 0.0, 0.0])
    if up is None:
        up = np.array([0.0, 0.0, 1.0])

    # build camera basis
    forward = center - eye
    forward = forward / np.linalg.norm(forward)

    right = np.cross(forward, up)
    right = right / np.linalg.norm(right)

    true_up = np.cross(right, forward)

    # rotation matrix (world → camera)
    R = np.vstack([right, true_up, forward])

    # transform + project
    V_cam = vertices_3d @ R.T
    vertices_2d = V_cam[:, :2]

    return vertices_2d

def vertex_angles(vertices, edges):
    # input vertices: (N,3) array of vertex positions and edges: list of (i,j) pairs of vertex indices
    # output: array of angles (in degrees) between edges at each vertex
    N = vertices.shape[0]

    # Build adjacency: neighbors for each vertex
    neighbors = [[] for _ in range(N)]
    for i, j in edges:
        neighbors[i].append(j)
        neighbors[j].append(i)

    angles = []

    for i in range(N):
        vi = vertices[i]
        nbrs = neighbors[i]
        v = [vertices[j] - vi for j in nbrs]

        for a in range(3): # Compute 3 pairwise angles
            for b in range(a+1, 3):
                va = v[a]
                vb = v[b]
    
                va = va / np.linalg.norm(va)
                vb = vb / np.linalg.norm(vb)
                
                cosang = np.clip(np.dot(va, vb), -1.0, 1.0) # Angle via dot product
                angle = np.arccos(cosang)

                angles.append(angle)

    angles = np.degrees(np.array(angles))

    return angles

def plot_structure_3d(vertices_3d_fit, vertices_3d_true, edges, eye):
    # Create 3D plot using plotly
    # Build line segments (with None separators)
    fig = plotly.graph_objects.Figure()

# --- true structure ---
    x_true, y_true, z_true = [], [], []

    for i, j in edges:
        x_true += [vertices_3d_true[i, 0], vertices_3d_true[j, 0], None]
        y_true += [vertices_3d_true[i, 1], vertices_3d_true[j, 1], None]
        z_true += [vertices_3d_true[i, 2], vertices_3d_true[j, 2], None]

    fig.add_trace(plotly.graph_objects.Scatter3d(
        x=x_true,
        y=y_true,
        z=z_true,
        mode='lines',
        line=dict(width=3, dash='solid', color='red'),
        name='True'
    ))

    # ---  fitted structure ---
    x_lines, y_lines, z_lines = [], [], []

    for i, j in edges:
        x_lines += [vertices_3d_fit[i, 0], vertices_3d_fit[j, 0], None]
        y_lines += [vertices_3d_fit[i, 1], vertices_3d_fit[j, 1], None]
        z_lines += [vertices_3d_fit[i, 2], vertices_3d_fit[j, 2], None]

    fig.add_trace(plotly.graph_objects.Scatter3d(
        x=x_lines,
        y=y_lines,
        z=z_lines,
        mode='lines',
        line=dict(width=5,dash='dot',color='blue'),
        name='Fitted'
    ))

    rng =2.0
    fig.update_layout(
        scene=dict(
            # Make axes equal for proper cube appearance
            aspectmode='cube', 
            # Remove axes ticks, labels, background and grid for a cleaner look
            xaxis=dict(range=[-rng, rng], showgrid=False, zeroline=False, showticklabels=False, title='', showbackground=False),
            yaxis=dict(range=[-rng, rng], showgrid=False, zeroline=False, showticklabels=False, title='', showbackground=False),
            zaxis=dict(range=[-rng, rng], showgrid=False, zeroline=False, showticklabels=False, title='', showbackground=False),
        ),
        scene_camera=dict(
            eye=dict(x=eye[0], y=eye[1], z=eye[2]),
            projection=dict(type="orthographic")
        ),
        margin=dict(l=0, r=0, b=0, t=0)
    )

    fig.show()


def negLogPosterior(x, vertices_2d_true, eye, edges):
    # x is a flattened version of the 3D vertex array, so reshape it to (-1, 3)
    vertices_3d_guess = x.reshape(-1, 3)
    
    #return ? 
    # Fill out the objective function to measure how well the 3D guess matches the observed 2D projection
    # then remove the NotImplementedError below
    raise NotImplementedError("Fill in the objective function")

# fit the model
vertices_2d_true_eye1 = project_orthographic(vertices_3d_true, eye1)

vertices_3d_init_guess = 1.5*(np.random.rand(*vertices_3d_true.shape)-0.5)

# Fit using single view
res = minimize(
negLogPosterior,
# flatten the initial guess to a 1D array for optimization
vertices_3d_init_guess.ravel(),
args=(vertices_2d_true_eye1, eye1, edges),
method="BFGS",
options={"maxiter": 10000}
)

# res.x is a flattened version of the 3D vertex array, so reshape it to (-1, 3)
vertices_3d_fit = res.x.reshape(-1, 3)

plot_structure_3d(vertices_3d_fit, vertices_3d_true, edges, eye1)

print("Converged:", res.success)
print("Message:", res.message)
print("2D projection error:", res.fun)




NotImplementedError: Fill in the objective function